# ModelID m_009


## Section 1 — Setup and imports

Content: import dependencies, set paths, and declare model metadata.

In [1]:
from pathlib import Path
import importlib.util
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

MODEL_ID = 'm_009'
BASE_DIR = Path.cwd().resolve().parent if Path.cwd().name == 'train_nb' else Path('model_training')
TRAIN_FACTOR = BASE_DIR / 'training_data' / 'train_df_factor.csv'
TRAIN_TARGET = BASE_DIR / 'training_data' / 'train_df_target.csv'
VAL_FACTOR = BASE_DIR / 'val_data' / 'val_df_factor.csv'
VAL_TARGET = BASE_DIR / 'val_data' / 'val_df_target.csv'
SCORING_SCRIPT = BASE_DIR / 'help_stuff' / 'validation_score.py'


## Section 2 — Data loading and required checks

Content: load train/validation datasets and run schema/quality checks required before training.

In [2]:
def load_data():
    X_train = pd.read_csv(TRAIN_FACTOR, low_memory=False)
    y_train = pd.read_csv(TRAIN_TARGET)['Prepaied_3m']
    X_val = pd.read_csv(VAL_FACTOR, low_memory=False)
    y_val = pd.read_csv(VAL_TARGET)['Prepaied_3m']
    return X_train, y_train, X_val, y_val


def run_checks(X_train, y_train, X_val, y_val):
    return {
        'train_factor_shape': X_train.shape,
        'train_target_shape': y_train.shape,
        'val_factor_shape': X_val.shape,
        'val_target_shape': y_val.shape,
        'schema_match': list(X_train.columns) == list(X_val.columns),
        'duplicate_columns_train': int(X_train.columns.duplicated().sum()),
        'duplicate_columns_val': int(X_val.columns.duplicated().sum()),
        'missing_cells_train': int(X_train.isna().sum().sum()),
        'missing_cells_val': int(X_val.isna().sum().sum()),
        'train_positive_rate': float(y_train.mean()),
        'val_positive_rate': float(y_val.mean()),
        'numeric_column_count': len(X_train.select_dtypes(include=['number', 'bool']).columns),
        'categorical_column_count': len(X_train.select_dtypes(exclude=['number', 'bool']).columns),
    }

X_train, y_train, X_val, y_val = load_data()
checks = run_checks(X_train, y_train, X_val, y_val)
checks

{'train_factor_shape': (45062, 74),
 'train_target_shape': (45062,),
 'val_factor_shape': (104, 74),
 'val_target_shape': (104,),
 'schema_match': True,
 'duplicate_columns_train': 0,
 'duplicate_columns_val': 0,
 'missing_cells_train': 1674885,
 'missing_cells_val': 3879,
 'train_positive_rate': 0.004127646353912388,
 'val_positive_rate': 0.5,
 'numeric_column_count': 55,
 'categorical_column_count': 19}

## Section 3 — Pipeline and grid search

Content: create preprocessing + classifier-agnostic pipeline (RandomForest and LogisticRegression), run GridSearchCV on training data, and select the best parameter set.

In [3]:
def drop_all_null_columns(X_train, X_val):
    all_null_cols = [col for col in X_train.columns if X_train[col].isna().all()]
    X_train2 = X_train.drop(columns=all_null_cols)
    X_val2 = X_val.drop(columns=all_null_cols)
    return X_train2, X_val2, all_null_cols


def build_pipeline(X_train):
    numeric_cols = X_train.select_dtypes(include=['number', 'bool']).columns.tolist()
    categorical_cols = [c for c in X_train.columns if c not in numeric_cols]

    preprocess = ColumnTransformer(
        transformers=[
            ('num', Pipeline([('imputer', SimpleImputer(strategy='median', keep_empty_features=True))]), numeric_cols),
            ('cat', Pipeline([
                ('imputer', SimpleImputer(strategy='most_frequent', keep_empty_features=True)),
                ('onehot', OneHotEncoder(handle_unknown='ignore'))
            ]), categorical_cols),
        ],
        remainder='drop'
    )

    return Pipeline([('preprocess', preprocess), ('clf', LogisticRegression(max_iter=1000))])


X_train_model, X_val_model, all_null_columns = drop_all_null_columns(X_train, X_val)
checks['all_null_columns_dropped'] = all_null_columns

param_grid = [
    {
        'clf': [RandomForestClassifier(random_state=42, n_jobs=1)],
        'clf__n_estimators': [120],
        'clf__max_depth': [3, 5],
        'clf__min_samples_leaf': [2],
        'clf__class_weight': ['balanced'],
    },
    {
        'clf': [LogisticRegression(max_iter=2000, solver='liblinear', random_state=42)],
        'clf__C': [1.0],
        'clf__class_weight': ['balanced'],
    }
]

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
pipeline = build_pipeline(X_train_model)
grid = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring='balanced_accuracy',
    n_jobs=1,
    cv=cv,
    verbose=1
)
grid.fit(X_train_model, y_train)
best_model = grid.best_estimator_
best_params = grid.best_params_
best_cv_score = grid.best_score_
best_params, best_cv_score


Fitting 3 folds for each of 3 candidates, totalling 9 fits


({'clf': RandomForestClassifier(n_jobs=1, random_state=42),
  'clf__class_weight': 'balanced',
  'clf__max_depth': 3,
  'clf__min_samples_leaf': 2,
  'clf__n_estimators': 120},
 np.float64(0.6753329001904861))

## Section 4 — Validation scoring (official script)

Content: score validation predictions with `help_stuff/validation_score.py` and print a compact result summary.

In [4]:
def load_validation_score_fn(script_path: Path):
    spec = importlib.util.spec_from_file_location('validation_score', script_path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod.prediction_score

prediction_score = load_validation_score_fn(SCORING_SCRIPT)
y_val_pred = best_model.predict(X_val_model)
val_score = prediction_score(y_val_pred)

result = {
    'model_id': MODEL_ID,
    'model_type': f"{type(best_model.named_steps['clf']).__name__} + GridSearchCV",
    'best_params': best_params,
    'best_cv_score': float(best_cv_score),
    'validation_score': float(val_score),
    'checks': checks,
}
result


{'model_id': 'm_009',
 'model_type': 'RandomForestClassifier + GridSearchCV',
 'best_params': {'clf': RandomForestClassifier(n_jobs=1, random_state=42),
  'clf__class_weight': 'balanced',
  'clf__max_depth': 3,
  'clf__min_samples_leaf': 2,
  'clf__n_estimators': 120},
 'best_cv_score': 0.6753329001904861,
 'validation_score': 0.7019230769230769,
 'checks': {'train_factor_shape': (45062, 74),
  'train_target_shape': (45062,),
  'val_factor_shape': (104, 74),
  'val_target_shape': (104,),
  'schema_match': True,
  'duplicate_columns_train': 0,
  'duplicate_columns_val': 0,
  'missing_cells_train': 1674885,
  'missing_cells_val': 3879,
  'train_positive_rate': 0.004127646353912388,
  'val_positive_rate': 0.5,
  'numeric_column_count': 55,
  'categorical_column_count': 19,
  'all_null_columns_dropped': ['UPB at Issuance',
   'Interest Only First Principal And Interest Payment Date',
   'Months to Amortization',
   'Mortgage Insurance Cancellation Indicator',
   'Scheduled Principal Curren